In [ ]:
""""""""""""""
#Our model contains the following key layers: cropping ->segmentation -> EfficientNetv2 -> GRU -> output
#For efficient preprocessing, we use seperate code for cropping and segmentation
#This part of the code takes cropped shots and perform segmentation layering on them

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


Mounted at /content/gdrive


In [ ]:
!pip install opencv-python-headless

In [ ]:
!pip install ultralytics opencv-python-headless

from ultralytics import YOLO

seg_model = YOLO("/content/gdrive/MyDrive/Shot Dataset/Other/Saved Training/Segmentation/Patience30FinalData/weights/best.pt") #segmentation model location

In [ ]:
import os
import cv2
import numpy as np
from google.colab import files

import cv2

def process_video(input_video_path, output_video_path, output_size=(224, 224)):
    """
    Process the video to detect and crop the 'Batsman' and save the cropped frames as a video.

    Parameters:
    - input_video_path (str): Path to the input video file.
    - output_video_path (str): Path to the output video file.
    - model (Model): The object detection model.
    - output_size (tuple): The size to which the cropped frames will be resized.
    """

    # Open the input video
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print("Error: Cannot open video file.")
        return

    # Get the frame rate of the input video
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Initialize the video writer for the output video
    fourcc = cv2.VideoWriter_fourcc('M','J','P','G')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, output_size)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Perform detection
        results = seg_model(frame, imgsz= 256, max_det=2, show_boxes=False, show_labels=False, show_conf=False, verbose=False)

        frame= results[0].plot(boxes=False, probs=False, labels=False)

        # Write the frame into the output file
        out.write(frame)

        # Release resources

    cap.release()
    out.release()
    cv2.destroyAllWindows()



def load_data(data_dir, output_dir):
    """
    Processes all videos in the given directory and saves the cropped videos to the output directory.

    Parameters:
    - data_dir (str): Path to the main data directory containing subdirectories of videos.
    - output_dir (str): Path to the directory where the output videos will be saved.
    """
    # Loop through all folders in the data directory
    for label in os.listdir(data_dir):
        label_folder = os.path.join(data_dir, label)

        # Ensure that it's a folder (not a file)
        if os.path.isdir(label_folder):
            output_label_dir = os.path.join(output_dir, label)
            os.makedirs(output_label_dir, exist_ok=True)

            print(f"Processing video: {label}")

            # Process all videos in the current folder
            for video in os.listdir(label_folder):
                video_path = os.path.join(label_folder, video)

                # Skip if not a video file (optional, depending on the content of the folder)
                if not video.endswith(('.mp4', '.avi', '.mov')):
                    continue

                video_name, _ = os.path.splitext(video)
                output_video_path = os.path.join(output_label_dir, f"{video_name}.avi")


                process_video(video_path, output_video_path)





In [ ]:
"Saves each segmented video in output locations"
#[change paths accordingly]

data_dir = '/content/gdrive/MyDrive/Shot Dataset/All Shots Cropped 15frames'
output_dir = '/content/gdrive/MyDrive/Shot Dataset/All Shots Cropped Segmented 15frames'
load_data(data_dir, output_dir)